<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 20 Views

The notebook judges the four views: the directions of the axes of the GZ lesson in each normal view
as `View()` of `3D_bau.frm` sets them up, the reference lines between the associated views, the way
back from a pixel to a point, the edges and outlines a view draws, the grid, the picking of a body,
the oblique view of version 1.3, the turn of the tab Kamera, the ladder of the zoom and, in the
page, the place of the origin in the classic views. A cell that proves an entry of the register
`Documentation/quirks.md` carries the tag its column Test names, and its checks name the ID of the
entry.

In [ ]:
width = 400
height = 300
require_browser = True  # tests/run.py passes False under --fast

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()

## 1. The axes of the normal views

The front view (Aufriss) shows y to the right and z up, the top view (Grundriss) y to the right and
x down, the side view (Kreuzriss) x to the left and z up, as the lesson draws them. A unit spans as
many pixels across as up, so the units are square, where the user scales of 1.3 make a vertical unit
of the Aufriss and the Kreuzriss 2 percent longer.

In [ ]:
def pixel(view, p):
    return devtool('project', view, *p, width, height)

def direction(view, axis):
    o = pixel(view, (0, 0, 0))
    e = pixel(view, axis)
    return (round(e[0] - o[0], 3), round(e[1] - o[1], 3))

ppu = 24.0  # the default scale of Ortho
want = {
    (FRONT, (0, 1, 0)): (ppu, 0), (FRONT, (0, 0, 1)): (0, -ppu), (FRONT, (1, 0, 0)): (0, 0),
    (TOP, (0, 1, 0)): (ppu, 0), (TOP, (1, 0, 0)): (0, ppu), (TOP, (0, 0, 1)): (0, 0),
    (SIDE, (1, 0, 0)): (-ppu, 0), (SIDE, (0, 0, 1)): (0, -ppu), (SIDE, (0, 1, 0)): (0, 0),
}
names = {FRONT: 'Aufriss', TOP: 'Grundriss', SIDE: 'Kreuzriss'}
for (view, axis), expected in want.items():
    got = direction(view, axis)
    check(close(got, expected, 1e-3), f'{names[view]}: the axis {axis} runs {expected} pixels per unit', got)
for view in (FRONT, TOP, SIDE):
    lengths = sorted(round(math.hypot(*direction(view, axis)), 3) for axis in ((1, 0, 0), (0, 1, 0), (0, 0, 1)))
    check(lengths[0] == 0 and lengths[1] == lengths[2] == ppu, f'Q-21 the units of the {names[view]} are square, {ppu} pixels across and up', lengths)

## 2. The reference lines

The three normal views share one centre and one scale, so a point lies at the same column in the
front and the top view and at the same row in the front and the side view, and the reference lines
join a marked point across the views.

In [ ]:
for p in [(1, 2, 3), (-4, 7.5, 0.5), (9, -3, 6)]:
    f, t, s = pixel(FRONT, p), pixel(TOP, p), pixel(SIDE, p)
    check(close(f[0], t[0], 1e-3), f'R-35 {p} lies on one vertical reference line in Aufriss and Grundriss', f, t)
    check(close(f[1], s[1], 1e-3), f'R-35 {p} lies on one horizontal reference line in Aufriss and Kreuzriss', f, s)

## 3. From a pixel back to a point

A parallel view answers the two coordinates it shows and nothing for the third, each view through
its own camera, where `xy_MouseDown` of 1.3 swaps the two coordinates of the Grundriss. Negative
values turn back as well, since every view shows every octant around its centre.

In [ ]:
for view, hidden in [(FRONT, 0), (SIDE, 1), (TOP, 2)]:
    p = (1.5, -2.0, 3.25)
    x, y = pixel(view, p)
    back = devtool('unproject', view, x, y, width, height)
    shown = [i for i in range(3) if i != hidden]
    check(back[hidden] is None and close([back[i] for i in shown], [p[i] for i in shown], 1e-3),
          f'Q-50 {names[view]} turns the pixel of {p} back into its two shown coordinates through its own camera', back)
    q = (-1.25, -2.5, -0.75)
    x, y = pixel(view, q)
    back = devtool('unproject', view, x, y, width, height)
    check(0 <= x <= width and 0 <= y <= height and close([back[i] for i in shown], [q[i] for i in shown], 1e-3),
          f'T-p77b {names[view]} shows the point {q} of negative coordinates and turns its pixel back', (x, y), back)

## 4. The edges and the outline

A view draws the edges where faces meet at an angle and the outline where a curved surface turns
away from the viewer. A cube shows its twelve edges, a cylinder seen from the front its two rims and
its two outer generators, and a sphere seen from the front the great circle x = 0. Version 1.3
removes no hidden line. The modern look shades the faces and draws the edges behind them a second
time, dashed and faint, through the pipeline whose override `HIDDEN` holds, and the classic look
draws every edge solid.

In [ ]:
cube = devtool('lines', packed(item('CUBE')), FRONT, width, height)
check(cube['styles']['edge'] == 12, 'the front view of a cube draws its twelve edges', cube['styles'])

cylinder = devtool('lines', packed(item('CYLINDER')), FRONT, width, height)
check(cylinder['styles']['edge'] == 2 * 48 + 2, 'the front view of a cylinder draws two rims and two generators', cylinder['styles'])

sphere = devtool('lines', packed(item('SPHERE')), FRONT, width, height)
points = [(s[0], s[1], s[2]) for s in sphere['segments']] + [(s[3], s[4], s[5]) for s in sphere['segments']]
on_circle = all(abs(x) < 1e-3 and abs(math.hypot(y, z) - 1) < 1e-3 for x, y, z in points)
check(sphere['count'] > 40 and on_circle, 'the outline of a sphere in the front view is its great circle x = 0', sphere['count'])

top = devtool('lines', packed(item('SPHERE')), TOP, width, height)
points = [(s[0], s[1], s[2]) for s in top['segments']]
check(all(abs(z) < 1e-3 for _, _, z in points), 'the outline of a sphere in the top view is its equator')

turned = packed(item('CUBE', rotation=(20, 30, 10)))
faces = devtool('faces', turned)
wgsl = (ROOT / 'crates' / 'bausteine-web' / 'src' / 'shaders' / 'scene.wgsl').read_text(encoding='utf-8')
gpu_rs = (ROOT / 'crates' / 'bausteine-web' / 'src' / 'gpu.rs').read_text(encoding='utf-8')
check(devtool('lines', turned, FRONT, width, height)['styles']['edge'] == 12 and faces['triangles'] == 12
      and 'override HIDDEN: bool' in wgsl and 'if HIDDEN {' in wgsl and 'if dashed ||' in wgsl and 'hidden: true' in gpu_rs,
      'T-p4b the modern look shades the twelve faces of a cube and draws all twelve edges, those behind the faces dashed by the pipeline HIDDEN', faces)
wires = devtool('classic-wires', turned, FRONT)
check(len(wires) == 12 and {w['style'] for w in wires} == {8} and 'if in.style >= 8u {' in wgsl,
      'T-p4b the classic look draws every edge of a cube solid, twelve wires of the style WIRE', len(wires))

## 5. A body cut away by a difference

The rasterizer does not compute the solid of the CSG, so a body a difference takes away draws as a
dashed ghost in the style `cutter`. The classic look draws the members of the group as plain wires,
and a declaration and its instance each draw the body they hold, where 1.3 draws no group at all.

In [ ]:
scene = packed(item('DIFFERENCE'), item('CUBOID', size=(4, 4, 2), parent=0), item('CYLINDER', (2, 2, -1), (1, 1, 4), parent=0))
styles = devtool('lines', scene, FRONT, width, height)['styles']
check(styles['edge'] == 12 and styles['cutter'] == 98, 'F4-Q59 the cuboid draws solid and the drill as a ghost', styles)
wires = devtool('classic-wires', scene, FRONT)
check(len(wires) == 12 + 2 * 12 + 12 and {w['style'] for w in wires} == {8}, 'F4-Q59 the classic look draws the blank and the drill of the difference as plain wires', len(wires))
placed = packed(item('DECLARE'), item('CUBE', parent=0), item('INSTANCE', (4, 0, 0), source=0))
check(devtool('lines', placed, FRONT, width, height)['styles']['edge'] == 24 and len(devtool('classic-wires', placed, FRONT)) == 24,
      'F4-Q59 a declaration and its instance each draw the cube they hold, in both looks')

## 6. The grid

A normal view shows the grid of its plane of projection over its viewport and the axes it shows.
Each axis runs as far as the grid lines along it, from the edge its negative half points to as far
as the edge of its positive half, so both halves of the axis cross the view (P-035), and an axis
that stopped at the origin would show. The oblique view shows the ground from zero to ten and the
axes with their arrows, as `D3_Achse` does.

In [ ]:
#: The codes of the styles of a segment, Style of drawing.rs.
GRID, AXIS = (3, 4), {'axis_x': 5, 'axis_y': 6, 'axis_z': 7}

def covers(spans, lo, hi):
    # Answers whether the spans (a, b) cover lo to hi without a gap and reach no further.
    reach = lo
    for a, b in sorted(spans):
        if a > reach + 1e-4:
            return False
        reach = max(reach, b)
    return abs(min(a for a, _ in spans) - lo) < 1e-4 and abs(reach - hi) < 1e-4

for view, axes in [(FRONT, ('axis_y', 'axis_z')), (TOP, ('axis_x', 'axis_y')), (SIDE, ('axis_x', 'axis_z'))]:
    drawn = devtool('grid', view, width, height)
    g = drawn['styles']
    shown = [a for a in ('axis_x', 'axis_y', 'axis_z') if g[a] > 0]
    check(tuple(shown) == axes and g['minor'] > 0 and g['major'] > 0, f'{names[view]} shows the grid and the axes {axes}', g)
    lines = [s for s in drawn['segments'] if s[6] in GRID]
    for axis in axes:
        i = 'xyz'.index(axis[-1])
        lo = min(min(s[i], s[i + 3]) for s in lines)
        hi = max(max(s[i], s[i + 3]) for s in lines)
        spans = [tuple(sorted((s[i], s[i + 3]))) for s in drawn['segments'] if s[6] == AXIS[axis]]
        on_axis = all(abs(s[j]) < 1e-4 and abs(s[j + 3]) < 1e-4 for s in drawn['segments'] if s[6] == AXIS[axis] for j in range(3) if j != i)
        check(spans and on_axis and covers(spans, lo, hi),
              f'P-035 the {axis[-1]} axis of the {names[view]} spans {lo} to {hi}, the extent of the grid lines along it', spans, (lo, hi))
        ends = [pixel(view, tuple(v if k == i else 0 for k in range(3))) for v in (lo, hi)]
        k = 0 if abs(ends[1][0] - ends[0][0]) > abs(ends[1][1] - ends[0][1]) else 1
        side = (width, height)[k]
        check(min(e[k] for e in ends) <= 0 and max(e[k] for e in ends) >= side,
              f'P-035 both halves of the {axis[-1]} axis of the {names[view]} reach the edges of the view of {width} by {height}', ends)
g = devtool('grid', OBLIQUE, width, height)['styles']
check(g['minor'] + g['major'] == 22 and g['axis_x'] == g['axis_y'] == g['axis_z'] == 3, 'the oblique view shows the ground grid and three arrows', g)

## 7. Picking a body

A click picks the body nearest to the viewer under the pixel, and nothing where no body lies.

In [ ]:
scene = packed(item('SPHERE', (2, 2, 1)), item('CUBE', (5, 1.5, 0.5)))
x, y = pixel(FRONT, (2, 2, 1))
check(devtool('pick', scene, FRONT, x, y, width, height) == 1, 'in the front view the cube in front of the sphere wins')
x, y = pixel(SIDE, (2, 2, 1))
check(devtool('pick', scene, SIDE, x, y, width, height) == 0, 'in the side view the sphere is hit')
check(devtool('pick', scene, FRONT, 2, 2, width, height) == -1, 'an empty corner picks nothing')

## 8. The oblique view of version 1.3

`D3_Refresh` of `Module1.bas` draws the 3D view as a cavalier projection: one unit of y runs 40
points to the right, one of z 40 points up, and one of x, towards the viewer, 20 points to the left
and 5 down, around the fixed centre (164, 230) of a picture box of 328 points. The classic look
draws its Schrägriss so, `BAUSTEINE_CAVALIER` gives the devtool that camera at a 3D zoom, and at the
start zoom of 1 a view of 328.27 points shows the numbers of 1.3 exactly. The origin follows the
actual view, at half its width and 70 percent of its height, and a unit spans 40 / 328.27 of its
height, where the DLL of 1.3 takes the fixed place (164, 230) for any size.

In [ ]:
cav = {'BAUSTEINE_CAVALIER': '1'}
side = 328.27
o = devtool('project', OBLIQUE, 0, 0, 0, side, side, env=cav)
check(close(o, (164.135, 229.789), 1e-3), 'the origin sits at the centre of D3_Refresh', o)
for axis, step in [((1, 0, 0), (-20, 5)), ((0, 1, 0), (40, 0)), ((0, 0, 1), (0, -40))]:
    e = devtool('project', OBLIQUE, *axis, side, side, env=cav)
    check(close((e[0] - o[0], e[1] - o[1]), step, 1e-3), f'Q-61 one unit of {axis} runs {step} points', e)
double = devtool('project', OBLIQUE, 0, 1, 0, side, side, env={'BAUSTEINE_CAVALIER': '2'})
check(close(double[0] - o[0], 80, 1e-3), 'twice the 3D zoom doubles the units', double)
o = devtool('project', OBLIQUE, 0, 0, 0, width, height, env=cav)
e = devtool('project', OBLIQUE, 0, 1, 0, width, height, env=cav)
check(close(o, (width / 2, 0.7 * height), 1e-3) and close(e[0] - o[0], 40 / 328.27 * height, 1e-3),
      f'Q-56 in a view of {width} by {height} the origin sits at half the width and 70 percent of the height, and a unit of y spans 40 / 328.27 of the height', o, e)

## 9. The dot raster of version 1.3

`grid` of `3D_bau.frm` draws a white dot at every unit of a normal view, and in the Aufriss and the
Kreuzriss one more at every half unit of the ground line. At 24 pixels a unit a view of 400 by 300
around the default centre (5, 5, 4) holds 18 by 14 points of the lattice, and the ground line 17
half steps.

In [ ]:
dots = {view: devtool('classic-raster', view, width, height)['dots'] for view in (FRONT, SIDE, TOP)}
check(dots[TOP] == 18 * 14 and 18 * ppu > width and 14 * ppu > height, 'Q-32 the Grundriss holds the lattice alone, over the whole view', dots)
check(dots[FRONT] == dots[SIDE] == 18 * 14 + 17, 'Q-32 the Aufriss and the Kreuzriss add the half steps of the ground line', dots)

## 10. The turn of the tab Kamera

The tab Kamera of 1.3 turns a copy of the scene in the 3D view and leaves the data alone. The
renderer turns the camera the other way, which gives the same picture, so a point turned by the
camera lands where the turned point lands in the camera without a turn. The turn is one matrix built
from the three angles for each frame, about the origin, so no error adds up, each axis turns once by
its own angle, and all three turn in the same sense, where the slider of x of 1.3 turns the other
way than those of y and z.

In [ ]:
import math
turn = (30, -20, 45)
def rotate(p, t):
    x, y, z = p
    for axis, deg in (('z', t[2]), ('y', t[1]), ('x', t[0])):
        c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
        if axis == 'x': y, z = c * y - s * z, s * y + c * z
        if axis == 'y': x, z = c * x + s * z, -s * x + c * z
        if axis == 'z': x, y = c * x - s * y, s * x + c * y
    return (x, y, z)
for env in ({}, cav):
    for p in [(1, 0, 0), (0, 2, 0), (1, 2, 3)]:
        turned = devtool('project', OBLIQUE, *p, width, height, env={**env, 'BAUSTEINE_TURN': ','.join(map(str, turn))})
        plain = devtool('project', OBLIQUE, *rotate(p, turn), width, height, env=env)
        check(close(turned, plain, 1e-2), f'M-57 the turned camera shows {p} as the turned point {"(cavalier)" if env else "(orbit)"}', turned, plain)

def shown(p, t, env=cav):
    return devtool('project', OBLIQUE, *p, width, height, env={**env, 'BAUSTEINE_TURN': ','.join(map(str, t))})
check(close(shown((0, 0, 0), turn), devtool('project', OBLIQUE, 0, 0, 0, width, height, env=cav), 1e-2), 'M-57 the turn is about the origin, which stays where it is')
tick = (2, 2, 0)
check(close(shown((1, 2, 3), tick), devtool('project', OBLIQUE, *rotate((1, 2, 3), tick), width, height, env=cav), 1e-2)
      and not close(shown((1, 2, 3), tick), devtool('project', OBLIQUE, *rotate((1, 2, 3), (4, 2, 0)), width, height, env=cav), 1e-2),
      'F4-Q88 a tick of two axes turns each by its own step once, and the first one not twice')
for p, t, q in [((0, 1, 0), (90, 0, 0), (0, 0, 1)), ((0, 0, 1), (0, 90, 0), (1, 0, 0)), ((1, 0, 0), (0, 0, 90), (0, 1, 0))]:
    check(close(shown(p, t), devtool('project', OBLIQUE, *q, width, height, env=cav), 1e-2), f'T-p62g a positive turn {t} takes {p} to {q}, so all three axes turn in the same sense')

## 11. The ladder of the zoom

`ZoomIn_Click` of `3D_bau.frm` multiplies `zoomf` by 1.2 while it is below 50 and `ZoomOut_Click`
divides it by 1.2 while it is above 5, from the 10 of `Form_Load`, which the automatic
`ZoomIn_Click` of the start turns into 12. The 3D zoom `d3zoomf` of F5 follows the same rule from
10. Each test lets one step pass its limit, so the zoom reaches 4.823 and 51.598, and `Timer1`
writes `zoomf / 10` into the field of F5, 0.48 to 5.16 where the thesis promises 50 to 500 percent.
The harness holds the ladder as `ZOOM_LADDER` and the text of the field as `zoom_text`, which the
notebooks 60 and 70 compare the page with. Version 1.3 computes in Single and writes up to seven
digits, and the page keeps the zoom on the exact rungs, so both texts agree to five digits on every
rung and differ in the seventh at two.

In [ ]:
import struct
rungs = ladder()
check([round(v, 3) for v in rungs] == ZOOM_LADDER, 'Q-28 the ladder times 1.2 below 50 and divided by 1.2 above 5 from 10 reaches the fourteen values of the notes', [round(v, 3) for v in rungs])
check(ladder(12.0) == ladder() or close(ladder(12.0), rungs, 1e-9), 'the start zoom 12 lies on the same ladder')
check(round(rungs[0] / 10, 2) == 0.48 and round(rungs[-1] / 10, 2) == 5.16, 'Q-28 each limit lets one step pass, the field shows 0.48 to 5.16 and not 0.5 to 5', rungs[0], rungs[-1])
check([zoom_text(v) for v in rungs[4:8]] == ['1', '1,2', '1,44', '1,728'] and zoom_text(rungs[2]) == '0,6944444',
      'Q-31 the classic field shows the raw zoomf / 10 with a comma, 1,2 and 1,728 and 0,6944444')
check(zoom_text(12, 'modern') == '120 %', 'Q-31 the modern field shows the zoom as a percentage, 120 %')

def single(x):
    return struct.unpack('f', struct.pack('f', x))[0]
def walk(direction):
    z, out = single(10.0), []
    while True:
        out.append(z)
        if not (z < 50 if direction > 0 else z > 5):
            return out
        z = single(z * 1.2) if direction > 0 else single(z / 1.2)
singles = sorted(set(walk(-1) + walk(1)))
texts13 = [f'{single(z / 10):.7g}'.replace('.', ',') for z in singles]
texts20 = [zoom_text(v) for v in rungs]
check(len(singles) == len(rungs) and all(a[:6] == b[:6] for a, b in zip(texts13, texts20)), 'the Single arithmetic of 1.3 and the exact rungs agree to five digits', texts13, texts20)
apart = [round(v, 3) for v, a, b in zip(rungs, texts13, texts20) if a != b]
check(apart == [35.832, 42.998], 'they differ in the seventh digit at 35.832 and 42.998 alone', apart)
print('zoomf    classic    modern    1.3 in Single')
for v, a, b in zip(rungs, texts20, texts13):
    print(f'{v:7.3f}  {a:10} {zoom_text(v, "modern"):9} {b}')

## 12. The origin of the classic views in the page

`ViewControl` of version 1.3 computes `x0 = y0 = F/5` in pixels, F the shorter outer side of Form1,
and applies the number in the user scale 328.27 of the picture boxes, so the origin sits at the
fraction (F/5) / 328.27 of each view: F is 500, 718 and 974 at the screens 800 by 600, 1024 by 768
and 1280 by 1024. The classic look of the page reproduces the picture with one centre shared by the
three normal views, so the origin lies that far from the edge each positive axis points away from:
the left and the bottom edge of the Aufriss, the right and the bottom edge of the Kreuzriss, the
left and the top edge of the Grundriss, at every zoom. Each viewport lies inside the bevel of its
view, 2 pixels on each side. The Schrägriss is the cavalier view of `D3_Refresh` with its origin at
half the width and 70 percent of the height, where a unit of y runs 40 points to the right, a unit
of z 40 points up and a unit of x 20 points to the left and 5 down, in a view of 328.27 points. The
notebook measures through `bausteine.app.bridge.project`, from the cameras of the last frame, in
headless Chromium, and leaves the section out under `--fast`. For the step of the zoom every answer
of the worker reaches the page 300 ms late, as on a loaded machine, and the harness waits with
`until` for the zoom and then for the frame after it, never for a fixed time.

In [ ]:
MEASURE = """() => {
  const b = bausteine.app.bridge, c = b.cameras;
  const form = document.querySelector('#form1').getBoundingClientRect();
  const views = [0, 1, 2, 3].map((v) => {
    const r = document.querySelector(`.view[data-view="${v}"]`).getBoundingClientRect();
    return { w: c[v * 18 + 16], h: c[v * 18 + 17], box: r.width, origin: b.project(v, 0, 0, 0) };
  });
  const o = b.project(3, 0, 0, 0);
  const units = [[1, 0, 0], [0, 1, 0], [0, 0, 1]].map(([x, y, z]) => { const p = b.project(3, x, y, z); return [p[0] - o[0], p[1] - o[1]]; });
  const front = b.project(0, 0, 1, 0)[0] - b.project(0, 0, 0, 0)[0];
  return { F: Math.min(form.width, form.height), zoomf: bausteine.app.zoomf, d3zoomf: bausteine.app.d3zoomf, views, units, front, open: b.pending.size };
}"""

#: Lets every message of the worker reach the page `delay` milliseconds late, in order, or at once for 0.
LATE = """(delay) => {
  const w = bausteine.app.bridge.worker;
  w.harnessReceive ??= w.onmessage;
  const receive = w.harnessReceive;
  w.onmessage = delay > 0 ? (e) => setTimeout(() => receive.call(w, e), delay) : receive;
}"""

def fractions(m):
    # The share of each normal view between the origin and the edge each positive axis points away from.
    (a, k, g) = m['views'][:3]
    return {
        'Aufriss': (a['origin'][0] / a['w'], 1 - a['origin'][1] / a['h']),
        'Kreuzriss': (1 - k['origin'][0] / k['w'], 1 - k['origin'][1] / k['h']),
        'Grundriss': (g['origin'][0] / g['w'], g['origin'][1] / g['h']),
    }

if require_browser:
    site = Site()
    try:
        for (w, h), F in [((800, 600), 500), ((1024, 768), 718), ((1280, 1024), 974)]:
            page = Browser(site.url, width=w, height=h, look='classic')
            try:
                page.open(screen=f'{w}x{h}')
                m = page.eval(MEASURE)
                want = F / 5 / 328.27
                side = m['views'][0]['w']
                check(m['F'] == F, f'at {w} by {h} Form1 has the shorter outer side F = {F}', m['F'])
                check(all(v['box'] - v['w'] == 4 for v in m['views']), f'at {w} by {h} every viewport lies inside the bevel of 2 pixels of its view', m['views'])
                for name, got in fractions(m).items():
                    check(close(got, (want, want), 1.0 / side), f'Q-20 at {w} by {h} the origin of the {name} lies at {want:.4f} of the view from its edges', got)
                page.eval(LATE, 300)
                page.eval("() => bausteine.act('zoom2d-in')")
                stepped = page.until('(z) => bausteine.app.zoomf > z', m['zoomf'])
                zoomed = page.eval(MEASURE)
                page.eval(LATE, 0)
                check(stepped and zoomed['open'] == 0 and close(zoomed['front'] / m['front'], zoomed['zoomf'] / m['zoomf'], 1e-3),
                      f'at {w} by {h} until() and settle() wait for the frame of a worker 300 ms late, whose unit grew with the zoom from {m["zoomf"]:g} to {zoomed["zoomf"]:g}',
                      stepped, zoomed['open'], m['front'], zoomed['front'])
                check(zoomed['zoomf'] > m['zoomf'] and all(close(a, b, 1.0 / side) for a, b in zip(fractions(zoomed).values(), fractions(m).values())),
                      f'at {w} by {h} the origin keeps its place when the zoom steps in', fractions(zoomed))
                o = m['views'][3]
                check(close(o['origin'], (o['w'] / 2, 0.7 * o['h']), 0.01), f'at {w} by {h} the origin of the Schrägriss sits at half the width and 70 percent of the height', o)
                unit = o['h'] / 328.27 * m['d3zoomf'] / 10
                got = [[round(c / unit, 3) for c in u] for u in m['units']]
                check(got == [[-20, 5], [40, 0], [0, -40]], f'at {w} by {h} the units of x, y and z run (-20, 5), (40, 0) and (0, -40) points of the Schrägriss', got)
                check(page.errors() == [], f'at {w} by {h} the page logs no error', page.errors())
            finally:
                page.close()
    finally:
        site.stop()
else:
    print('the run has no browser, the section stands aside')